# 5.0 Colonisation parameter effects: how recipe and conditions change colonisation (plotly)

**Question:** when a study changed one parameter (substrate, particle size, nanoclay, recipe), how did colonisation change?

**Read this first.** Each colonisation outcome we have comes from **one study only** (photo coverage from Elsacker 2019, coverage from Elsacker 2022, colonisation score from Musci 2025, end points from two more studies, yield from Pacheco-Sanchez 2026). Pooling studies would mostly show lab differences, like comparing plants grown in different greenhouses. So section 2 puts all studies on one chart but marks each source, and you should compare within a source first.

1. Overview heatmap: which parameters change inside which study
2. Colonisation over days across studies, by C:N (lines, and days to full colonisation against C:N)
3. Pacheco-Sanchez 2026: yield (biological efficiency) by substrate mix and C:N

**How to use a plotly chart:** hover for details; drag a box to zoom, double-click to zoom out; click a legend name to hide it, double-click to show only that one.

Needs plotly: run `pip install plotly` once in the .venv before Run All (same as 0.3). Section 2 reads `data/interim/coverage_auto.csv`, written by notebook 2.4.

In [7]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from mycodemo.config import INTERIM_DATA_DIR

EXTRACTED_DIR = INTERIM_DATA_DIR / "extracted"

NAMES = {
    "d002": "Musci 2025", "d003": "Gomaa 2024", "d004": "Bhaskar 2026", "d005": "Elsacker 2019",
    "d006": "Hysek 2025", "d007": "Irbe 2026", "d009": "Elsacker 2022", "d012": "Pacheco-Sanchez 2026",
    "d016": "Vašatko 2022", "d017": "Jiménez-Obando 2025", "d023": "Houette 2022",
}

## Data

One row per sample: `sample.csv` joined to `fermentation.csv` (fungus, water, spawn, temperature, days, treatments) and `substrate.csv` (plant, particle size, C:N, pre-treatment). `measurement.csv` holds the outcomes.

In [8]:
sample = pd.read_csv(EXTRACTED_DIR / "sample.csv")
ferm = pd.read_csv(EXTRACTED_DIR / "fermentation.csv")
substrate = pd.read_csv(EXTRACTED_DIR / "substrate.csv")
meas = pd.read_csv(EXTRACTED_DIR / "measurement.csv")

rows = (sample.merge(ferm, on="fermentation_id", how="left")
        .merge(substrate, on="substrate_id", how="left"))
rows["study"] = rows["dataset_id"].map(NAMES)
rows.groupby("study").size().rename("samples").to_frame()

,samples
study,
Bhaskar 2026,58
Elsacker 2019,93
Elsacker 2022,5
Gomaa 2024,17
Houette 2022,12
Hysek 2025,91
Irbe 2026,23
Jiménez-Obando 2025,6
Musci 2025,7


## 1. Overview: which parameters change inside each study

**Rows** are parameters, grouped as substrate, growth (fermentation) and treatments. **Columns** are studies.
**The number in a cell** is how many different values that parameter takes among the study's samples.

- **1 (pale)**: held fixed in that study. You cannot learn its effect from that study.
- **2 or more (dark)**: the study changed it, so its effect can be looked at *inside* that study.
- **blank**: not reported.

How to read it like a biologist: a dark cell is an experiment the authors ran for you. A parameter that is pale in every column but takes different values *between* columns (for example temperature) was never tested on its own. Its effect is tangled with the lab.

Pre-treatment is free text, so its count partly reflects how each paper describes the recipe.

In [9]:
PARAMS = {
    "Substrate": ["plant_name", "particle_size_mm", "cn_estimate", "pre_treatment", "sterilisation"],
    "Growth": ["species", "water_pct_wt", "spawn_pct_wt", "incubation_temp_c", "rh_pct",
               "total_growth_days", "container_id", "fill_height_mm"],
    "Treatments": ["in_process_treatment", "post_treatment", "post_treatment_temp_c"],
}
labels, counts = [], []
for group, cols in PARAMS.items():
    for col in cols:
        n = rows.groupby("study")[col].nunique()                 # distinct values per study
        reported = rows.groupby("study")[col].apply(lambda s: s.notna().any())
        counts.append(n.where(reported))                          # blank when never reported
        labels.append(f"{group}: {col}")
grid = pd.DataFrame(counts, index=labels)
grid = grid[[NAMES[k] for k in NAMES if NAMES[k] in grid.columns]]

fig = px.imshow(grid, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                zmin=0, zmax=max(4, grid.max().max()),
                labels=dict(x="Study", y="Parameter", color="distinct values"),
                title="How many values each parameter takes inside each study (1 = held fixed)")
fig.update_xaxes(side="top", tickangle=-30)
fig.update_layout(height=650)
fig.show()

### 1b. Parameter against parameter: which parameters change together

Both axes are parameters (plus **study** as its own row, so you can see what is tangled with the lab).
**The number in a cell** (Cramér's V, 0 to 1) says how strongly knowing one parameter tells you the other, across all samples where both are reported:

- **near 0 (pale)**: the two change independently. Their effects could be told apart.
- **near 1 (dark)**: they always change together, like two genes that are always inherited together. If colonisation changes, you cannot tell which one caused it.
- **blank**: fewer than two values in the samples that report both, so nothing to compare.

Look first at the **study** row: a dark cell there means that parameter mostly differs *between* labs, not inside one.
On our data most cells are dark: almost every parameter is tangled with the study, which is why section 2 marks the source of every line and asks you to compare within a source first.

Numbers are treated as categories (e.g. 22, 26, 28 °C are three groups), because most parameters take only a few values.

In [10]:
import numpy as np


def cramers_v(a, b):
    """Association between two categorical columns, 0 = independent, 1 = always together (bias-corrected)."""
    table = pd.crosstab(a, b)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    n = table.values.sum()
    expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / n
    chi2 = ((table.values - expected) ** 2 / expected).sum()
    r, k = table.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_c, k_c = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    denom = min(k_c - 1, r_c - 1)
    return np.sqrt(phi2 / denom) if denom > 0 else np.nan


cols = ["study"] + [c for group in PARAMS.values() for c in group]
cat = rows[cols].astype(str).where(rows[cols].notna())   # numbers as categories, missing stays missing
assoc = pd.DataFrame(np.nan, index=cols, columns=cols)
for i, a in enumerate(cols):
    if cat[a].nunique() > 1:
        assoc.loc[a, a] = 1.0
    for b in cols[i + 1:]:
        both = cat[[a, b]].dropna()
        assoc.loc[a, b] = assoc.loc[b, a] = cramers_v(both[a], both[b])

fig = px.imshow(assoc.round(2), text_auto=True, aspect="auto", color_continuous_scale="Blues",
                zmin=0, zmax=1, labels=dict(x="Parameter", y="Parameter", color="Cramér's V"),
                title="Which parameters change together across all samples (1 = always together)")
fig.update_xaxes(side="top", tickangle=-45)
fig.update_layout(height=750)
fig.show()

## 2. Colonisation over days across studies, by C:N

Five sources measured how fast the substrate was colonised, each in its own way. This section puts them in one table (`colonisation`) with one row per sample and day:

| kind | source | what the number is |
|---|---|---|
| photo | Elsacker 2019 (d005) | share of the dish that looks white in a photo, × 100 |
| read from figure | Elsacker 2022 (d009) | surface colonised (%), mean of 3 dishes, read off the paper's figure |
| score estimate (rough) | Musci 2025 (d002) | colonisation score 1 to 5 converted to %, see below |
| end point only | Gebru 2024 (d025), Akcay 2023 (d026) | only the day the surface was fully (100 %) colonised |

**Musci score to %:** the paper gives a score, not a percentage. We turn it into % by straight-line interpolation between the middle of each band: 1 = 2.5 %, 2 = 18 %, 3 = 50 %, 4 = 82 %, 5 = 97.5 %. Treat it as a rough guide only.

In [11]:
SCORE_MIDPOINTS = {1: 2.5, 2: 18, 3: 50, 4: 82, 5: 97.5}
SERIES_TO_SAMPLE = {
    "d005-Flax": "d005-S-FL-PD-1", "d005-W": "d005-S-WL-PD-1", "d005-FD": "d005-S-FD-PD-1",
    "d005-FSW": "d005-S-FWL-PD-1", "d005-H": "d005-S-HL-PD-1", "d005-S": "d005-S-SL-PD-1",
    "d005-FLT": "d005-S-FTT-PD-1", "d005-FLU": "d005-S-FUT-PD-1",
}


def score_to_pct(score):
    """Colonisation score 1 to 5 -> % colonised, straight line between band midpoints."""
    return np.interp(score, list(SCORE_MIDPOINTS), list(SCORE_MIDPOINTS.values()))


def build_colonisation_table():
    """One long table: source, sample_id, species, day, pct_colonised, cn_estimate, cn_confidence, kind."""
    info = rows[["sample_id", "genus", "species", "plant_name", "cn_estimate", "cn_confidence"]].copy()
    info["species"] = info["genus"].str[0] + ". " + info["species"]
    info = info.drop(columns="genus")

    photo = pd.read_csv(INTERIM_DATA_DIR / "coverage_auto.csv")
    photo["sample_id"] = photo["series_id"].map(SERIES_TO_SAMPLE)
    photo = photo.assign(dataset_id="d005", pct_colonised=photo["coverage"] * 100, kind="photo")

    fig_read = meas[(meas["dataset_id"] == "d009") & meas["surface_colonised_pct"].notna()]
    fig_read = fig_read.assign(day=fig_read["day_of_growth"], pct_colonised=fig_read["surface_colonised_pct"],
                               kind="read from figure")

    musci = meas[(meas["dataset_id"] == "d002") & meas["colonisation_score"].notna()]
    musci = musci.assign(day=musci["day_of_growth"], pct_colonised=score_to_pct(musci["colonisation_score"]),
                         kind="score estimate (rough)")

    full = meas[meas["dataset_id"].isin(["d025", "d026"]) & (meas["surface_colonised_pct"] == 100)]
    full = (full.groupby(["dataset_id", "sample_id"], as_index=False)["day_of_growth"].min()   # earliest day at 100 %
            .rename(columns={"day_of_growth": "day"}).assign(pct_colonised=100.0, kind="end point only"))

    keep = ["dataset_id", "sample_id", "day", "pct_colonised", "kind"]
    long = pd.concat([photo[keep], fig_read[keep], musci[keep], full[keep]], ignore_index=True)
    long = long.merge(info, on="sample_id", how="left")
    long["source"] = long["dataset_id"].map({**NAMES, "d025": "Gebru 2024", "d026": "Akcay 2023"})
    long["recipe"] = long["sample_id"].str.split("-", n=2).str[-1] + ": " + long["plant_name"]
    cols = ["source", "sample_id", "species", "day", "pct_colonised", "cn_estimate", "cn_confidence", "kind", "recipe"]
    return long[cols].dropna(subset=["day", "pct_colonised"]).sort_values(["source", "sample_id", "day"]).reset_index(drop=True)


colonisation = build_colonisation_table()
print(colonisation["cn_estimate"].isna().sum(), "of", len(colonisation), "rows have no C:N estimate")
colonisation.groupby(["source", "kind"]).agg(samples=("sample_id", "nunique"), rows=("day", "size"))

0 of 209 rows have no C:N estimate


,,samples,rows
source,kind,,
Akcay 2023,end point only,13,13
Elsacker 2019,photo,8,67
Elsacker 2022,read from figure,5,45
Gebru 2024,end point only,7,7
Musci 2025,score estimate (rough),7,77


### Panel A: % colonised over days

Each line is one sample (one dish or bag). **Colour** = estimated C:N (log scale), **dash** = how the number was obtained (dashed = rough score conversion), **marker shape** = source. Hover shows source, recipe, species, C:N and how confident the C:N estimate is.

- Dishes and bags differ in speed, so a line from a Petri dish and a line from a bag are not directly comparable.
- Musci is a rough conversion from a score, not a measurement.
- C:N values are estimates (see the confidence in the hover), not lab measurements.
- Compare within a source first, then look across.

**Dropdown:** switch the x axis between days and days ÷ the source's own days to full colonisation (median over that source's samples; if none reached full colonisation, as in Elsacker 2019, the median last day is used instead). In the second view 1.0 means "as fast as a typical sample of that source", which removes the dish-versus-bag speed difference.

In [12]:
SYMBOLS = {s: sym for s, sym in zip(sorted(colonisation["source"].unique()),
                                   ["circle", "square", "diamond", "triangle-up", "cross", "x"])}
DASH = {"photo": "solid", "read from figure": "solid", "score estimate (rough)": "dash", "end point only": "solid"}


def days_to_full(df, threshold=95):
    """Per sample: first day >= threshold % (or the end point); samples that never get there keep their last day."""
    out = []
    for (source, sid), g in df.sort_values("day").groupby(["source", "sample_id"]):
        hit = g[g["pct_colonised"] >= threshold]
        reached = len(hit) > 0
        out.append(dict(source=source, sample_id=sid, species=g["species"].iloc[0], recipe=g["recipe"].iloc[0],
                        cn_estimate=g["cn_estimate"].iloc[0], cn_confidence=g["cn_confidence"].iloc[0],
                        kind=g["kind"].iloc[0], days_full=(hit if reached else g)["day"].iloc[0] if reached
                        else g["day"].iloc[-1], reached=reached))
    return pd.DataFrame(out)


full_days = days_to_full(colonisation)
own_days = full_days[full_days["reached"]].groupby("source")["days_full"].median()   # each source's own scale
never = sorted(set(full_days["source"]) - set(own_days.index))
for source in never:   # no sample reached 95 %: use the median last day seen (a lower bound) so the source still shows
    own_days[source] = full_days.loc[full_days["source"] == source, "days_full"].median()
print("No sample reached 95 % in:", ", ".join(never) or "none", "(their scale is the median last day, a lower bound)")
colonisation["day_rel"] = colonisation["day"] / colonisation["source"].map(own_days)
print("Median days to full colonisation per source:\n", own_days.round(1).to_string())

cn_ok = colonisation["cn_estimate"].dropna()
lo, hi = np.log10(cn_ok.min()), np.log10(cn_ok.max())


def cn_colour(cn):
    if pd.isna(cn):
        return "#999999"
    t = (np.log10(cn) - lo) / (hi - lo) if hi > lo else 0.5
    return px.colors.sample_colorscale("Viridis", [float(t)])[0]


groups = list(colonisation.groupby("sample_id", sort=False))
fig = go.Figure()
for sid, g in groups:
    r = g.iloc[0]
    cn_txt = "not reported" if pd.isna(r["cn_estimate"]) else f"{r['cn_estimate']:g}"
    fig.add_trace(go.Scatter(
        x=g["day"], y=g["pct_colonised"], mode="lines+markers", name=r["source"], legendgroup=r["source"],
        showlegend=False, line=dict(color=cn_colour(r["cn_estimate"]), dash=DASH[r["kind"]], width=2),
        marker=dict(symbol=SYMBOLS[r["source"]], size=8, color=cn_colour(r["cn_estimate"])),
        customdata=[[r["source"], r["recipe"], r["species"], cn_txt, str(r["cn_confidence"]), r["kind"]]] * len(g),
        hovertemplate=("<b>%{customdata[0]}</b> (%{customdata[5]})<br>recipe: %{customdata[1]}<br>"
                       "species: %{customdata[2]}<br>C:N: %{customdata[3]} (confidence: %{customdata[4]})<br>"
                       "day %{x:.3g}: %{y:.1f} % colonised<extra></extra>")))
# legend entries for the marker shapes
for source, sym in SYMBOLS.items():
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=source, legendgroup=source,
                             marker=dict(symbol=sym, size=9, color="#444444")))
# colour bar for C:N (log)
ticks = [t for t in [10, 20, 30, 50, 100, 200] if cn_ok.min() * 0.8 <= t <= cn_ok.max() * 1.2]
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                         marker=dict(colorscale="Viridis", cmin=lo, cmax=hi, color=[lo], showscale=True,
                                     colorbar=dict(title="C:N (log)", tickvals=[np.log10(t) for t in ticks],
                                                   ticktext=[str(t) for t in ticks], len=0.7))))
days_abs = [g["day"].tolist() for _, g in groups]
days_rel = [g["day_rel"].tolist() for _, g in groups]
n_extra = len(SYMBOLS) + 1
pad = [[None]] * n_extra
fig.update_layout(
    title="Colonisation over days across studies (colour = C:N, dashed = rough score conversion, grey = no C:N)",
    xaxis_title="Day", yaxis_title="Colonised (%)", height=620,
    updatemenus=[dict(type="dropdown", x=0.0, y=1.15, xanchor="left", buttons=[
        dict(label="x = days", method="update",
             args=[{"x": days_abs + pad}, {"xaxis.title.text": "Day"}]),
        dict(label="x = days ÷ source's days to full colonisation", method="update",
             args=[{"x": days_rel + pad}, {"xaxis.title.text": "Days ÷ the source's median days to full colonisation"}]),
    ])])
fig.update_yaxes(range=[0, 105])
fig.show()

No sample reached 95 % in: Elsacker 2019 (their scale is the median last day, a lower bound)
Median days to full colonisation per source:
 source
Akcay 2023       25.0
Elsacker 2022     7.0
Gebru 2024       15.3
Musci 2025       21.0
Elsacker 2019    10.0


### Panel B: days to full colonisation against C:N

One dot per sample. Days to full colonisation = the first day with 95 % or more colonised, or the day of the end point. **Symbol** = source, **colour** = species. An **open marker** is a sample that never reached 95 %; it is drawn at its last day, so its true time is longer.

- Dishes and bags differ in speed, so compare dots of the same source first.
- Musci days come from a rough score conversion.
- C:N values are estimates; samples without one are left out of this panel.

In [13]:
species_colour = dict(zip(sorted(full_days["species"].dropna().unique()), px.colors.qualitative.Set2))
plot_b = full_days.dropna(subset=["cn_estimate"])
print(len(full_days) - len(plot_b), "samples without a C:N estimate are left out of panel B")

fig = go.Figure()
for (source, species, reached), g in plot_b.groupby(["source", "species", "reached"]):
    fig.add_trace(go.Scatter(
        x=g["cn_estimate"], y=g["days_full"], mode="markers",
        name=f"{source}, {species}" + ("" if reached else " (not yet 95 %)"), legendgroup=source,
        marker=dict(symbol=SYMBOLS[source] + ("" if reached else "-open"), size=11,
                    color=species_colour[species], line=dict(width=2, color=species_colour[species])),
        customdata=g[["source", "recipe", "species", "kind", "cn_confidence"]].astype(str).values,
        hovertemplate=("<b>%{customdata[0]}</b> (%{customdata[3]})<br>recipe: %{customdata[1]}<br>"
                       "species: %{customdata[2]}<br>C:N %{x:g} (confidence: %{customdata[4]})<br>"
                       "day %{y:.3g}" + ("" if reached else " (last day seen, not yet 95 %)") + "<extra></extra>")))
fig.update_layout(title="Days to full colonisation against C:N (open marker = never reached 95 %)",
                  xaxis_title="Estimated C:N (log)", yaxis_title="Days to full colonisation", height=560)
fig.update_xaxes(type="log")
fig.show()

0 samples without a C:N estimate are left out of panel B


## 3. Pacheco-Sanchez 2026: yield by substrate mix and C:N

48 *P. djamor* bags. Biological efficiency (BE) = fresh mushrooms ÷ dry substrate × 100. One dot per bag.
Left: dots against estimated C:N, coloured by substrate mix. Right: the same bags as boxes per mix.

In [14]:
d012 = meas[meas["dataset_id"] == "d012"].merge(
    rows[["sample_id", "plant_name", "cn_estimate", "substrate_id"]], on="sample_id")
fig = px.scatter(d012, x="cn_estimate", y="biological_efficiency_pct", color="plant_name",
                 hover_data=["sample_id", "substrate_id"],
                 labels=dict(cn_estimate="Estimated C:N", biological_efficiency_pct="Biological efficiency (%)",
                             plant_name="Substrate mix"),
                 title="Pacheco-Sanchez 2026: yield against C:N, one dot per bag")
fig.show()

fig = px.box(d012, x="plant_name", y="biological_efficiency_pct", points="all",
             hover_data=["sample_id", "cn_estimate"],
             labels=dict(plant_name="Substrate mix", biological_efficiency_pct="Biological efficiency (%)"),
             title="Pacheco-Sanchez 2026: yield per substrate mix")
fig.update_xaxes(tickangle=-30)
fig.show()